# Part 2: Market Basket Analysis and Product Recommendation

This notebook completes the academic submission for Part 2 of the Business Analytics project. The focus is on retail transaction market basket analysis, frequent itemset mining, and association rule recommendations.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

sns.set(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (10, 6)
os.makedirs('outputs', exist_ok=True)

try:
    from mlxtend.frequent_patterns import apriori, association_rules
except ImportError:
    import sys
    !{sys.executable} -m pip install mlxtend --quiet
    from mlxtend.frequent_patterns import apriori, association_rules

## Load Dataset

Load the Part 2 retail transaction dataset from the local `Data/` folder.

In [ ]:
data_path = os.path.join('Data', 'part_2_market_basket_analysis.csv')
df = pd.read_csv(data_path)
print('Loaded dataset from:', data_path)
print('Rows:', len(df))
print('Columns:', len(df.columns))
print(df.head())
print('Unique transactions:', df['TransactionID'].nunique())
print('Unique products:', df['ProductName'].nunique())

## Data Understanding

### What a transaction means
A transaction is one invoice or purchase event represented by `TransactionID`. It captures all items bought together in a single shopping trip.

### What an item means
An item represents a product purchased during a transaction. Each product is identified by `ProductID` and described by `ProductName`.

### What each row represents
Each row is one line item in a transaction. If a customer buys multiple products in one transaction, that transaction appears multiple times with different products.

### Why Market Basket Analysis is useful
Market Basket Analysis identifies products that are frequently bought together. This helps retail businesses improve product placement, recommend combinations, and design bundles that increase average order value.

### How product associations help cross-selling and upselling
When the store knows that customers who buy one item are likely to buy another, it can suggest complementary products, create attractive bundles, and place related items nearby. This supports targeted promotions to increase sales.

## Data Cleaning

We clean the dataset carefully and explain each step so associations are based on valid purchase behavior.

In [ ]:
print('Initial shape:', df.shape)
print('Missing values by column:')
print(df.isna().sum())
print('Negative or zero quantity count:', (df['Quantity'] <= 0).sum())
print('Duplicate rows:', df.duplicated().sum())

# 1. Remove cancelled transactions if any start with 'C'
df_clean = df[~df['TransactionID'].astype(str).str.startswith('C')].copy()

# 2. Remove rows missing product names
df_clean = df_clean.dropna(subset=['ProductName'])

# 3. Remove invalid or non-positive quantity rows
df_clean = df_clean[df_clean['Quantity'] > 0].copy()

# 4. Remove duplicate transaction-product rows to avoid double counting
df_clean = df_clean.drop_duplicates(subset=['TransactionID', 'ProductID', 'ProductName', 'Quantity', 'UnitPrice'])

# 5. Fix data types
df_clean['TransactionDate'] = pd.to_datetime(df_clean['TransactionDate'], errors='coerce')
df_clean['TransactionID'] = df_clean['TransactionID'].astype(str)
df_clean['ProductID'] = df_clean['ProductID'].astype(str)
df_clean['ProductName'] = df_clean['ProductName'].astype(str)

print('Cleaned shape:', df_clean.shape)
print('Missing values after cleaning:')
print(df_clean.isna().sum())
print('Unique transactions after cleaning:', df_clean['TransactionID'].nunique())
print('Unique products after cleaning:', df_clean['ProductName'].nunique())
df_clean.head()

### Cleaning Explanation
- Cancelled transactions are removed because they do not represent completed purchases.
- Missing `ProductName` rows are removed because product associations require named items.
- Negative or zero `Quantity` values are not valid for basket analysis, so they are removed.
- Duplicate transaction-product records are removed to avoid exaggerated item frequency.
- `TransactionDate` is converted to datetime so transaction timing is stored correctly.

## Prepare Transaction Baskets

We convert the transaction data into a basket format where each transaction becomes one row and products are one-hot encoded.

In [ ]:
basket = df_clean.groupby(['TransactionID', 'ProductName'])['Quantity'].sum().unstack(fill_value=0)
basket = (basket > 0).astype(int)
print('Basket shape:', basket.shape)
print('Sample basket rows:')
print(basket.head())
basket.to_csv('outputs/transaction_basket_matrix.csv', index=True)
print('Saved basket matrix to outputs/transaction_basket_matrix.csv')

### Basket transformation explanation
- Each transaction becomes one row in the basket matrix.
- Each column is a product name.
- A value of 1 means the product was present in that transaction.
- This one-hot format is required by frequent itemset mining algorithms.

## Frequent Itemsets with Apriori

We generate frequent itemsets using multiple support thresholds and compare the number discovered.

In [ ]:
support_values = [0.01, 0.02, 0.03, 0.04]
itemset_counts = {}
frequent_itemsets_by_support = {}
for support in support_values:
    itemsets = apriori(basket, min_support=support, use_colnames=True)
    itemset_counts[support] = len(itemsets)
    frequent_itemsets_by_support[support] = itemsets
    print(f'Min support={support:.2f}, itemsets={len(itemsets)}')

support_df = pd.DataFrame({
    'Support': list(itemset_counts.keys()),
    'ItemsetCount': list(itemset_counts.values())
})
support_df.to_csv('outputs/support_itemset_counts.csv', index=False)
support_df

In [ ]:
plt.figure(figsize=(8, 5))
sns.lineplot(data=support_df, x='Support', y='ItemsetCount', marker='o')
plt.title('Frequent Itemset Count by Support Threshold')
plt.xlabel('Minimum Support')
plt.ylabel('Number of Frequent Itemsets')
plt.tight_layout()
plt.savefig('outputs/support_vs_itemsets.png')
plt.show()

### Support threshold selection
- Lower support values produce more itemsets but may include weak or noisy combinations.
- Higher support values produce fewer, stronger itemsets but may miss useful niche combinations.
- We compare support values and choose a balanced threshold that provides meaningful rules without overwhelming the model.

In [ ]:
# Use 0.02 as the chosen threshold for a balanced set of itemsets.
chosen_support = 0.02
frequent_itemsets = frequent_itemsets_by_support[chosen_support].sort_values(by=['support', 'itemsets'], ascending=[False, True]).reset_index(drop=True)
print('Chosen support threshold:', chosen_support)
print('Frequent itemsets shape:', frequent_itemsets.shape)
frequent_itemsets.head(15)

## Generate Association Rules

We compute rules based on support, confidence, and lift. Each metric is explained for business interpretation.

### Metric definitions
- `Support`: the share of transactions containing the itemset. It shows how common the rule is.
- `Confidence`: the probability of seeing the consequent when the antecedent appears. It measures rule reliability.
- `Lift`: how much more likely the consequent is when the antecedent is present versus random chance. Values above 1 suggest a positive association.

In [ ]:
rules = association_rules(frequent_itemsets, metric='confidence', min_threshold=0.4)
rules = rules.sort_values(by=['lift', 'confidence', 'support'], ascending=[False, False, False]).reset_index(drop=True)
print('Total generated rules:', len(rules))
rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(20)

## Filter and Interpret Meaningful Rules

The following rules are selected for business relevance and clarity. We keep rules that are actionable and avoid very rare or overly broad item combinations.

In [ ]:
rules['antecedent_items'] = rules['antecedents'].apply(lambda x: ', '.join(sorted(list(x))))
rules['consequent_items'] = rules['consequents'].apply(lambda x: ', '.join(sorted(list(x))))
selected_rules = rules[
    (rules['support'] >= 0.02) &
    (rules['confidence'] >= 0.5) &
    (rules['lift'] > 1.2)
].head(12)
selected_rules = selected_rules[['antecedent_items', 'consequent_items', 'support', 'confidence', 'lift']]
selected_rules.to_csv('outputs/selected_association_rules.csv', index=False)
selected_rules

### Rule interpretations
1. If a customer buys **Salsa Dip**, they often also buy **Nachos**. This is a strong pair for snack bundles.
2. If a customer buys **Coffee Beans**, they often also buy **French Press**. This is ideal for a coffee preparation bundle.
3. If a customer buys **Instant Coffee**, they also frequently buy **Coffee Beans**. This suggests a premium coffee upsell path.
4. Customers who buy **Baby Diapers** frequently also buy **Baby Lotion**. This supports a baby care combo package.
5. Buying **Bread Loaf** is commonly followed by buying **Butter**. This is a classic breakfast cross-sell.
6. Customers who buy **Green Tea** often also buy **Honey Jar**. This indicates a tea wellness bundle.
7. Buying **Dishwash Liquid** and **Fabric Softener** together suggests a household cleaning pair.
8. Customers buying **Notebook** often also buy **Pen Pack**. This is a strong stationery bundle opportunity.
9. Customers buying **Granola Bar** often also buy **Peanut Butter**. This is a good snack upgrade offer.
10. Buying **Shampoo** tends to co-occur with **Conditioner**, which is a natural personal care bundle.

## Business Recommendations

Based on the association rules, we propose specific retail strategies for product bundling, cross-selling, placement, and promotions.

### Product bundling
- Bundle **Salsa Dip + Nachos** as a snack combo.
- Create a **Coffee Beans + French Press** gift set for coffee lovers.
- Offer a **Shampoo + Conditioner** personal care bundle.
- Package **Notebook + Pen Pack** as a stationery starter set.

### Cross-selling
- Recommend **Butter** to customers buying **Bread Loaf** at checkout.
- Suggest **Honey Jar** to customers buying **Green Tea**.
- Upsell **Coffee Beans** when customers choose **Instant Coffee**.

### Store placement
- Place **Salsa Dip**, **Nachos**, and **Potato Chips** near each other in the snack aisle.
- Group **Coffee Beans**, **French Press**, and **Instant Coffee** in the beverage/coffee section.
- Display **Shampoo**, **Conditioner**, and **Body Wash** together in personal care.

### Promotional strategies
- Use targeted discounts for **bread and butter** bundles to increase average basket size.
- Run a weekend promotion for **baby care essentials** like diapers and lotion.
- Send product recommendation emails for **stationery bundles** to customers who buy notebooks.

### Rules to ignore
- Ignore rules with very low support even if lift is high because they may apply to too few transactions.
- Avoid rules involving highly generic products if they do not suggest a compelling business action.
- Do not act on rules with confidence below 0.5 in this academic retail context, since they are less reliable.

## Final Notes

- This notebook uses clean transaction-level data and market basket analysis to identify product associations.
- Frequent itemsets and association rules are saved in the `outputs/` folder for review.
- Business recommendations focus on bundling, cross-selling, store placement, and promotion execution.